# AI-Based Employee Attrition Prediction Using Machine Learning

**Final Year B.Tech / M.Tech Project**

This notebook predicts whether an employee is **Likely to Leave** or **Likely to Stay** using the IBM HR Analytics Employee Attrition & Performance dataset.

### Algorithms
- Logistic Regression
- Decision Tree
- Random Forest
- XGBoost

### Main Modules
Data Collection → Preprocessing → Feature Selection → Model Training → Prediction → Evaluation → Dashboard/Visualization → Explainability


## 1. Project Objective

Employee attrition can increase recruitment, training and productivity costs. The objective of this project is to build a classification system that estimates employee attrition risk from HR-related factors such as age, department, job role, income, overtime, satisfaction, work-life balance, distance from home and performance.

> **Important:** A prediction is a risk estimate, not a statement that an employee will definitely resign. HR decisions should use the model as decision support and should not be based on the prediction alone.


In [ ]:
# 2. Import Libraries
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)


## 3. Data Collection

Place `WA_Fn-UseC_-HR-Employee-Attrition.csv` in the same folder as this notebook.

The notebook also contains an optional URL fallback. If internet access is unavailable, download the CSV from Kaggle and place it beside the notebook.


In [ ]:
# 3. Load IBM HR Analytics dataset
DATA_FILE = "WA_Fn-UseC_-HR-Employee-Attrition.csv"

if os.path.exists(DATA_FILE):
    df = pd.read_csv(DATA_FILE)
else:
    # Public mirror of the IBM HR Analytics dataset.
    DATA_URL = "https://raw.githubusercontent.com/mragpavank/ibm-hr-analytics-attrition-dataset/master/WA_Fn-UseC_-HR-Employee-Attrition.csv"
    try:
        df = pd.read_csv(DATA_URL)
        print("Dataset loaded from public mirror.")
    except Exception as e:
        raise FileNotFoundError(
            "Dataset not found. Download WA_Fn-UseC_-HR-Employee-Attrition.csv "
            "from Kaggle and place it in the notebook folder."
        ) from e

print("Shape:", df.shape)
display(df.head())


In [ ]:
# 4. Basic Data Inspection
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).to_frame("missing"))

print("\nDuplicate rows:", df.duplicated().sum())


## 5. Exploratory Data Analysis

The target column is `Attrition`, with `Yes` indicating attrition and `No` indicating staying.


In [ ]:
# Attrition distribution
plt.figure(figsize=(7,5))
sns.countplot(data=df, x="Attrition")
plt.title("Employee Attrition Distribution")
plt.xlabel("Attrition")
plt.ylabel("Number of Employees")
plt.show()

print("Attrition percentage:")
display((df["Attrition"].value_counts(normalize=True) * 100).round(2).to_frame("Percentage"))


In [ ]:
# Selected project-focused visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df, x="OverTime", hue="Attrition", ax=axes[0,0])
axes[0,0].set_title("Overtime vs Attrition")

sns.boxplot(data=df, x="Attrition", y="MonthlyIncome", ax=axes[0,1])
axes[0,1].set_title("Monthly Income vs Attrition")

sns.countplot(data=df, x="JobSatisfaction", hue="Attrition", ax=axes[1,0])
axes[1,0].set_title("Job Satisfaction vs Attrition")

sns.boxplot(data=df, x="Attrition", y="DistanceFromHome", ax=axes[1,1])
axes[1,1].set_title("Distance From Home vs Attrition")

plt.tight_layout()
plt.show()


In [ ]:
# Department and Job Role attrition rates
dept_rate = pd.crosstab(df["Department"], df["Attrition"], normalize="index") * 100
role_rate = pd.crosstab(df["JobRole"], df["Attrition"], normalize="index") * 100

print("Attrition rate by Department (%)")
display(dept_rate.round(2))

print("Attrition rate by Job Role (%)")
display(role_rate.round(2))


## 6. Feature Selection

The proposed project specifically focuses on the following predictors:

- Age
- Department
- Job Role
- Monthly Income
- Years at Company
- Overtime
- Job Satisfaction
- Work-Life Balance
- Distance from Home
- Performance Rating

`Attrition` is the target variable.


In [ ]:
# Project feature set
FEATURES = [
    "Age",
    "Department",
    "JobRole",
    "MonthlyIncome",
    "YearsAtCompany",
    "OverTime",
    "JobSatisfaction",
    "WorkLifeBalance",
    "DistanceFromHome",
    "PerformanceRating"
]
TARGET = "Attrition"

missing_features = [c for c in FEATURES + [TARGET] if c not in df.columns]
if missing_features:
    raise ValueError(f"Required columns missing from dataset: {missing_features}")

X = df[FEATURES].copy()
y = df[TARGET].map({"No": 0, "Yes": 1})

print("Selected features:")
print(FEATURES)
print("\nTarget distribution:")
display(y.value_counts().rename(index={0:"Stay", 1:"Leave"}).to_frame("count"))


In [ ]:
# 7. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

numeric_features = [
    "Age", "MonthlyIncome", "YearsAtCompany",
    "JobSatisfaction", "WorkLifeBalance",
    "DistanceFromHome", "PerformanceRating"
]
categorical_features = ["Department", "JobRole", "OverTime"]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 8. Machine Learning Model Training

Four classification algorithms are compared:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. XGBoost

XGBoost is optional. If it is not installed, the notebook will continue with the three scikit-learn models and clearly report that XGBoost was skipped.


In [ ]:
# 8. Define models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        max_depth=10,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=RANDOM_STATE
    )
    print("XGBoost available.")
except ImportError:
    print("XGBoost is not installed. Run: pip install xgboost")


In [ ]:
# 9. Train and Evaluate All Models
trained_models = {}
results = []
predictions = {}
probabilities = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    trained_models[name] = pipe
    predictions[name] = y_pred
    probabilities[name] = y_prob

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).sort_values("ROC AUC", ascending=False).reset_index(drop=True)
display(results_df.style.format({
    "Accuracy":"{:.3f}", "Precision":"{:.3f}", "Recall":"{:.3f}",
    "F1 Score":"{:.3f}", "ROC AUC":"{:.3f}"
}))


## 10. Model Comparison


In [ ]:
# Model comparison chart
plot_df = results_df.set_index("Model")[["Accuracy", "Precision", "Recall", "F1 Score", "ROC AUC"]]

ax = plot_df.plot(kind="bar", figsize=(14,6))
ax.set_title("Machine Learning Model Performance Comparison")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
plt.xticks(rotation=20)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


In [ ]:
# 11. Confusion Matrix for the Best Model
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
best_pred = predictions[best_model_name]

cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(6,5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Likely Stay", "Likely Leave"],
    yticklabels=["Likely Stay", "Likely Leave"]
)
plt.title(f"Confusion Matrix - {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

print(classification_report(
    y_test, best_pred,
    target_names=["Likely Stay", "Likely Leave"],
    zero_division=0
))


In [ ]:
# 12. ROC Curves
plt.figure(figsize=(9,7))

for name in trained_models:
    fpr, tpr, _ = roc_curve(y_test, probabilities[name])
    auc = roc_auc_score(y_test, probabilities[name])
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0,1], [0,1], "--")
plt.title("ROC Curve Comparison")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 13. Important Factors Affecting Attrition

For tree-based models, feature importance is extracted from the trained estimator after preprocessing. This shows which transformed input variables contribute most strongly to the model's predictions.

This is model-based importance and should not be interpreted as proof of causation.


In [ ]:
# Feature importance for Random Forest / best tree model
importance_model_name = "Random Forest" if "Random Forest" in trained_models else best_model_name
pipe = trained_models[importance_model_name]

pre = pipe.named_steps["preprocessor"]
model = pipe.named_steps["model"]

feature_names = pre.get_feature_names_out()

if hasattr(model, "feature_importances_"):
    importances = model.feature_importances_
    fi = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    }).sort_values("Importance", ascending=False)

    display(fi.head(15))

    plt.figure(figsize=(10,7))
    sns.barplot(data=fi.head(15), y="Feature", x="Importance")
    plt.title(f"Top Attrition Factors - {importance_model_name}")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()
else:
    print(f"{importance_model_name} does not expose tree feature_importances_.")


## 14. Employee Attrition Prediction

Enter employee information below to generate:

- Prediction: **Likely to Leave** / **Likely to Stay**
- Probability of leaving
- Probability of staying
- Basic risk interpretation

The selected input fields match the project proposal.


In [ ]:
# 14. Prediction function
def predict_employee(
    age,
    department,
    job_role,
    monthly_income,
    years_at_company,
    overtime,
    job_satisfaction,
    work_life_balance,
    distance_from_home,
    performance_rating
):
    employee = pd.DataFrame([{
        "Age": age,
        "Department": department,
        "JobRole": job_role,
        "MonthlyIncome": monthly_income,
        "YearsAtCompany": years_at_company,
        "OverTime": overtime,
        "JobSatisfaction": job_satisfaction,
        "WorkLifeBalance": work_life_balance,
        "DistanceFromHome": distance_from_home,
        "PerformanceRating": performance_rating
    }])

    prob_leave = float(best_model.predict_proba(employee)[0, 1])
    prediction = "Likely to Leave" if prob_leave >= 0.50 else "Likely to Stay"

    if prob_leave >= 0.70:
        risk = "High Risk"
    elif prob_leave >= 0.40:
        risk = "Medium Risk"
    else:
        risk = "Low Risk"

    return {
        "Prediction": prediction,
        "Probability of Leaving": round(prob_leave, 4),
        "Probability of Staying": round(1 - prob_leave, 4),
        "Risk Level": risk
    }

# Example employee
example_prediction = predict_employee(
    age=30,
    department="Sales",
    job_role="Sales Executive",
    monthly_income=3500,
    years_at_company=3,
    overtime="Yes",
    job_satisfaction=2,
    work_life_balance=2,
    distance_from_home=15,
    performance_rating=3
)

display(pd.DataFrame([example_prediction]))


## 15. Simple HR Dashboard

The following section provides a notebook-based dashboard using Plotly. It summarizes the dataset and shows the model's performance and employee attrition distribution.


In [ ]:
# 15. Dashboard
try:
    import plotly.express as px
    import plotly.graph_objects as go
    from IPython.display import display

    dashboard_fig1 = px.pie(
        df, names="Attrition",
        title="Overall Employee Attrition"
    )
    dashboard_fig1.show()

    dashboard_fig2 = px.bar(
        dept_rate.reset_index(),
        x="Department",
        y="Yes",
        title="Attrition Rate by Department",
        labels={"Yes":"Attrition Rate (%)"}
    )
    dashboard_fig2.show()

    dashboard_fig3 = px.bar(
        results_df,
        x="Model",
        y="ROC AUC",
        title="Model ROC AUC Comparison",
        range_y=[0,1]
    )
    dashboard_fig3.show()

except ImportError:
    print("Plotly is not installed. Run: pip install plotly")


## 16. Optional SHAP Explainability

SHAP can be added as a future/advanced module to explain individual predictions. It is intentionally optional because SHAP support can differ between model versions and environments.


In [ ]:
# Optional SHAP section
# Uncomment after installing SHAP:
# pip install shap

try:
    import shap
    print("SHAP is installed.")
    print("For a production version, use SHAP TreeExplainer for the trained tree model.")
except ImportError:
    print("SHAP is not installed. Run: pip install shap")


## 17. Save the Best Model

The trained model pipeline contains preprocessing and the classifier, so it can be saved and reused for deployment.


In [ ]:
# 17. Save model and evaluation results
import joblib

joblib.dump(best_model, "employee_attrition_best_model.pkl")
results_df.to_csv("employee_attrition_model_comparison.csv", index=False)

print("Saved:")
print("- employee_attrition_best_model.pkl")
print("- employee_attrition_model_comparison.csv")
print("- Best model:", best_model_name)


## 18. Final Project Output

The completed implementation provides:

1. Employee attrition dataset loading
2. Data inspection and preprocessing
3. Exploratory visualization
4. Feature selection
5. Logistic Regression
6. Decision Tree
7. Random Forest
8. XGBoost when installed
9. Accuracy, Precision, Recall, F1 Score and ROC AUC
10. Confusion matrix
11. ROC curve comparison
12. Important attrition factors
13. Individual employee prediction
14. Leave/Stay probability
15. Risk-level indication
16. HR dashboard visualizations
17. Model export for deployment

### Future Enhancement
- Deep Learning using LSTM/Transformer
- SHAP-based Explainable AI
- Real-time HR dashboard
- HRMS integration
- Periodic model retraining
